In [2]:
from build_context import stringify_report, format_triplet, construct_query_context
from config import *
from langchain_community.graphs import Neo4jGraph
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Neo4jVector
from neo4j import GraphDatabase, Result
from typing import Dict, Any
import torch
import pandas as pd
import helpers

/home/damon/kg_aug_causal_disc_exp/build_context.py:12: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(NEO4J_URI,NEO4J_USERNAME,NEO4J_PASSWORD,NEO4J_DATABASE, refresh_schema=False)


In [3]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD),database=NEO4J_DATABASE)

def db_query(cypher: str, params: Dict[str, Any] = {}) -> pd.DataFrame:
    """Executes a Cypher statement and returns a DataFrame"""
    return driver.execute_query(
        cypher, parameters_=params, result_transformer_=Result.to_df
    )

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    refresh_schema=False,
    driver_config={"notifications_disabled_classifications": ["DEPRECATION"]}
)

embedding = HuggingFaceEmbeddings(
    model_name="pritamdeka/S-PubMedBert-MS-MARCO",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

/shared/graphrag/lib/python3.10/site-packages/langchain_community/graphs/neo4j_graph.py:404: PreviewWarning: notifications_disabled_classifications is a preview feature. It might be changed without following the deprecation policy. See also https://github.com/neo4j/neo4j-python-driver/wiki/preview-features.
  self._driver = neo4j.GraphDatabase.driver(
/tmp/ipykernel_893692/1428407539.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


In [4]:
# Fetch all relationships with their entity information
query = """
MATCH (source:__Entity__)-[r]->(target:__Entity__)
RETURN
    elementId(r) as rel_id,
    {
        start_id: source.id,
        start_desc: source.description,
        rel_desc: r.description,
        rel_type: type(r),
        end_id: target.id,
        end_desc: target.description,
        degree: apoc.node.degree(source) + apoc.node.degree(target)
    } as triplet
"""

relationships_df = db_query(query)
print(f"Found {len(relationships_df)} relationships")
relationships_df.head()

Found 27379 relationships


,rel_id,triplet
0,5:648f6927-f8b3-4a8c-9cc3-da9930e251f1:6,"{'degree': 1246, 'end_id': 'Coronary Artery Di..."
1,5:648f6927-f8b3-4a8c-9cc3-da9930e251f1:95,"{'degree': 1324, 'end_id': 'Cad', 'end_desc': ..."
2,5:648f6927-f8b3-4a8c-9cc3-da9930e251f1:35,"{'degree': 1324, 'end_id': 'Cad', 'end_desc': ..."
3,5:648f6927-f8b3-4a8c-9cc3-da9930e251f1:44715,"{'degree': 1230, 'end_id': 'New Illness', 'end..."
4,5:648f6927-f8b3-4a8c-9cc3-da9930e251f1:43965,"{'degree': 1229, 'end_id': 'Stomach_And_Abdomi..."


In [5]:
# Format relationships as text using the existing format_triplet function
from tqdm import tqdm

rel_texts = []
for _, row in tqdm(relationships_df.iterrows(), total=len(relationships_df), desc="Formatting triplets"):
    text = format_triplet(row['triplet'])
    rel_texts.append(text)

print(f"Sample formatted triplet:\n{rel_texts[0]}")
print(f"\nTotal formatted: {len(rel_texts)}")

Formatting triplets: 100%|██████████| 27379/27379 [00:00<00:00, 32054.10it/s]

Sample formatted triplet:
(Depression: Affective and cognitive manifestations of depression, which may be associated with various health outcomes, including heart disease.)--[CAUSALITY: Genetic liability to depression was significantly associated with an increased risk of Coronary Artery Disease in patients with diabetes.]-->(Coronary Artery Disease (Cad): A condition in which the coronary arteries become narrowed or blocked, reducing blood flow to the heart.)


Total formatted: 27379


In [6]:
# Generate embeddings using the embedding model
# We'll process in batches for efficiency
batch_size = 128
all_embeddings = []

for i in tqdm(range(0, len(rel_texts), batch_size), desc="Generating embeddings"):
    batch = rel_texts[i:i + batch_size]
    batch_embeddings = embedding.embed_documents(batch)
    all_embeddings.extend(batch_embeddings)

print(f"Generated {len(all_embeddings)} embeddings")
print(f"Embedding dimension: {len(all_embeddings[0])}")

Generating embeddings: 100%|██████████| 214/214 [00:47<00:00,  4.51it/s]

Generated 27379 embeddings
Embedding dimension: 768


In [7]:
# Store embeddings back to Neo4j relationships
store_query = """
UNWIND $data as row
MATCH ()-[r]->()
WHERE elementId(r) = row.rel_id
SET r.embedding = row.embedding
"""

# Process in batches to avoid memory issues
batch_size = 1000
for i in tqdm(range(0, len(relationships_df), batch_size), desc="Storing embeddings"):
    batch_data = [
        {
            "rel_id": relationships_df.iloc[j]['rel_id'],
            "embedding": all_embeddings[j]
        }
        for j in range(i, min(i + batch_size, len(relationships_df)))
    ]
    db_query(store_query, {"data": batch_data})

print("Done! Embeddings stored on relationships.")

Storing embeddings: 100%|██████████| 28/28 [00:47<00:00,  1.69s/it]

Done! Embeddings stored on relationships.


In [8]:
# Verify embeddings were stored
verify_query = """
MATCH ()-[r]->()
WHERE r.embedding IS NOT NULL
RETURN count(r) as count_with_embeddings
"""

result = db_query(verify_query)
print(f"Relationships with embeddings: {result.iloc[0]['count_with_embeddings']}")

Relationships with embeddings: 27379
